To run this, press "*Run*" and press "*Run All*" on your local **Intel GPU**!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
# Torch comes from the PyTorch XPU index. A plain `pip install unsloth` would pull CUDA torch.
!pip install --upgrade -qqq uv
!uv pip install -qqq "torch==2.13.0" "torchvision==0.28.0" --index-url https://download.pytorch.org/whl/xpu
# torch is already satisfied, so uv keeps the +xpu wheel.
!uv pip install -qqq "bitsandbytes>=0.50.0" unsloth unsloth_zoo
!uv pip install -qqq --no-deps "torchao==0.18.0" --index-url https://download.pytorch.org/whl/xpu
!uv pip install -qqq --no-deps -U "tokenizers>=0.22.0,<=0.23.0"


In [ ]:
# Disable expandable segments for PyTorch memory allocator on Intel XPU to enable multi-device workload
%env PYTORCH_ALLOC_CONF=expandable_segments:False

!uv pip install -qqq --upgrade --force-reinstall --no-deps git+https://github.com/unslothai/unsloth-zoo git+https://github.com/unslothai/unsloth
!uv pip install -qqq "transformers==4.57.6"
!uv pip install -qqq --upgrade --no-deps "trl==0.22.2"

!uv pip uninstall -qqq triton
!uv pip install -qqq --reinstall --no-deps "triton-xpu==3.7.2" --index-url https://download.pytorch.org/whl/xpu


env: PYTORCH_ALLOC_CONF=expandable_segments:False


### Unsloth

# Goal: Make faster kernels with Reinforcement Learning

Our goal is to make a faster matrix multiplication kernel by doing RL on GPT-OSS 20B with Unsloth.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Matrix_multiplication_qtl1.svg/500px-Matrix_multiplication_qtl1.svg.png" height=200 />

You will learn how to:
1. Counteract **reward hacking** like cheating, caching, laziness.
2. Timing and correctness of kernels and time limits.
3. Making good **reward functions**
4. How to seriously do RL to make optimized kernels

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 768 # Can increase for longer RL output
lora_rank = 4 # Larger rank = smarter, but slower
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b-BF16",
    device_map = "balanced", # Split the model across every Intel GPU
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    offload_embedding = False, # Reduces VRAM by 1GB
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sdp/joshua/.unsloth-nb/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1110 17:53:25.266000 1834692 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 17:53:25.297000 1834692 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: temporary patch patch_GptOssModel failed in phase pre_compile and was skipped. (ImportError: cannot import name 'is_flex_attention_decoding' from 'unsloth_zoo.flex_attention' (/home/sdp/joshua/.unsloth-nb/lib/python3.12/site-packages/unsloth_zoo/flex_attention/__init__.py))
[unsloth_zoo.log|WARNING]Unsloth: temporary patch patch_GptOssModel failed in phase post_compile and was skipped. (ImportError: cannot import name 'is_flex_attention_decoding' from 'unsloth_zoo.flex_attention' (/home/sdp/joshua/.unsloth-nb/lib/python3.12/site-packages/unsloth_zoo/flex_attention/__init__.py))


==((====))==  Unsloth 2026.11.4: Fast Gpt_Oss patching. Transformers: 4.57.6.
   \\   /|    Intel(R) Arc(TM) Pro B60 Graphics. Num GPUs = 4. Max memory: 23.906 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+xpu. Intel Toolkit: 20260000. Triton: 3.7.2
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 9/9 [00:18<00:00,  2.03s/it]


We now add some small amount of LoRA weights to GPT-OSS so we only need to train those, instead of training on the full model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Detected MoE model with num_experts = 32 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']


# Optimized matrix multiplication

Numpy has optimized matrix multiplication kernels for CPUs via BLAS optimized operations. For GPUs, one can use CUDA accelerated cuBLAS kernels which PyTorch calls under the hood.

To generate some random matrices to do matrix multiplication, we can do the below:

In [ ]:
import numpy as np
def generate_random_matrices(seed = 3407, n = 256):
    random_state = np.random.RandomState(seed)
    n, k, m = random_state.randint(1, n+1, size = 3)
    A = np.random.uniform(-10, 10, size = (n, k))
    B = np.random.uniform(-10, 10, size = (k, m))
    return A, A.tolist(), B, B.tolist()

We shall generate a small matrix, and see the matrix multiplied output

In [ ]:
A, A_list, B, B_list = generate_random_matrices(seed = 42, n = 5)
print(A)
print(B)
print(np.matmul(A, B))

[[-2.8313286   4.54613909 -7.95265309  6.53459836  2.87235103]
 [ 7.0739631   3.76278879  9.31565599 -8.52884711  9.96832952]
 [ 8.41214082  6.51136046 -3.79347975 -2.46773693 -2.32292989]
 [ 3.91302932  4.98335304 -5.33855089  5.71057634 -2.79871647]]
[[ 0.39218774 -9.6181377  -3.49736707]
 [-0.33354865 -1.05626139  3.87231208]
 [ 0.49494174  5.91863954 -6.83183693]
 [ 5.1465162  -7.51648113  1.00445384]
 [ 9.63213377 -4.92327556  3.323014  ]]
[[  54.73441488  -87.89725072   97.94605887]
 [  58.25238906   -1.8467447   -49.25453031]
 [ -35.82528794  -80.25394462   11.51225408]
 [  -0.33785799 -103.64132345   38.51974367]]


We can call a LLM to generate a simple matrix multiply kernel in Python only, and we can calculate the differences between the actual result and the kernel's result

In [ ]:
def calculate_difference(pred, real):
    if pred is None: return 5, 5
    assert real is not None
    import numpy as np
    try:
        difference = pred - real
    except:
        return 5, 5
    amax_error = float(np.amax(difference))
    mse_error  = float(np.mean(np.square(difference)))
    return amax_error, mse_error

In [ ]:
# Kernel generated by GPT-5
def matmul(A, B):
    z, s = zip, sum
    Bt = list(z(*B))
    return [[s(a*b for a, b in z(row, col)) for col in Bt] for row in A]

We see the error below is very small, so that's good!

In [ ]:
prediction = matmul(A_list, B_list)
calculate_difference(prediction, np.matmul(A, B))

(7.105427357601002e-15, 4.6783406255758477e-29)

# Countering Reward Hacking

The ultimate goal of RL is to maximize some reward (say speed, revenue, some metric).

But RL can **cheat** When the RL algorithm learns a trick or exploits something to increase the reward, without actually doing the task at end, this is called "Reward Hacking".

Some good examples are in https://en.wikipedia.org/wiki/Reward_hacking

For matrix multiplication kernels, we might see the following issues:

* Laziness: RL learns to use Numpy, Torch, other libraries, which calls optimized kernels.
* Caching: RL learns to cache the result of the output
* Cheating: RL learns to find the actual output by inspecting Python global variables
* RL learns to edit the timing function to make it output 0 time as passed.

And possibly more. We shall try to address each!

# Countering Reward Hacking 1: Stop laziness
We can stop the RL algorithm from calling optimized code by inspecting if the generated code imports other non standard Python libraries. We used GPT-5 to help generate this check `check_only_stdlib_imports`:

In [ ]:
#@title (Collapsible code)
import ast
import sys
import sysconfig
from pathlib import Path

def _stdlib_names():
    """
    Build a set of canonical stdlib top-level module/package names.
    Uses sys.stdlib_module_names when available (3.10+), with a
    filesystem fallback for older versions/edge cases.
    """
    names = {m.lower() for m in getattr(sys, "stdlib_module_names", set())}
    names |= {m.lower() for m in sys.builtin_module_names}
    names.add("__future__")  # special-case

    # Fallback/augmentation: scan the stdlib directory
    try:
        stdlib_dir = Path(sysconfig.get_path("stdlib"))
        if stdlib_dir.exists():
            for p in stdlib_dir.iterdir():
                if p.name == "site-packages":
                    continue
                if p.suffix == ".py":
                    names.add(p.stem.lower())
                elif p.is_dir() and (p / "__init__.py").exists():
                    names.add(p.name.lower())
    except Exception:
        # conservative fallback; the names set above will still work well
        pass

    return names

_STDLIB_SET = _stdlib_names()

def check_only_stdlib_imports(code: str):
    """
    Return (ok: bool, details: dict)

    ok == True  -> all absolute imports are from the stdlib.
    ok == False -> details['non_stdlib'] lists offending top-level modules.

    details includes:
      - stdlib: sorted list of stdlib imports found
      - non_stdlib: sorted list of non-stdlib imports found
      - relative_imports: count of relative imports (always allowed here)
    """
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return False, {
            "error": f"SyntaxError: {e}",
            "stdlib": [],
            "non_stdlib": [],
            "relative_imports": 0,
        }

    abs_imports = set()
    relative_count = 0

    class Visitor(ast.NodeVisitor):
        def visit_Import(self, node: ast.Import):
            for alias in node.names:
                abs_imports.add(alias.name.split(".")[0])
        def visit_ImportFrom(self, node: ast.ImportFrom):
            nonlocal relative_count
            if (node.level or 0) > 0:
                # relative import
                relative_count += 1
            else:
                if node.module:
                    abs_imports.add(node.module.split(".")[0])

    Visitor().visit(tree)

    stdlib_found = sorted(m for m in abs_imports if m.lower() in _STDLIB_SET)
    non_stdlib = sorted(m for m in abs_imports if m.lower() not in _STDLIB_SET)

    return len(non_stdlib) == 0, {
        "stdlib": stdlib_found,
        "non_stdlib": non_stdlib,
        "relative_imports": relative_count,
    }

For example, let's call `check_only_stdlib_imports` on a random piece of matrix multiplication code generated by GPT-5:

In [ ]:
sample = """
def matmul(A, B):
    import numpy as np
    from torch import matmul
    z, s = zip, sum
    Bt = list(z(*B))
    return [[s(a*b for a, b in z(row, col)) for col in Bt] for row in A]
"""
ok, info = check_only_stdlib_imports(sample)
print("Only stdlib imports?", ok)
print(info)

Only stdlib imports? False
{'stdlib': [], 'non_stdlib': ['numpy', 'torch'], 'relative_imports': 0}


# Countering Reward Hacking 2: Stop cheating
We can stop the RL algorithm from using global or cached variables by restricting it's `locals` and `globals`.

We are also going to use `exec` to create the function, so we have to save the output to an empty dict.

We also disallow global variable access.

In [ ]:
output_function = {}
exec(sample, {}, output_function)
output_function["matmul"]

<function matmul(A, B)>

We also disallow global variable access via `types.FunctionType(f.__code__, {})`

In [ ]:
import types
output_function["matmul"] = types.FunctionType(output_function["matmul"].__code__, {})

def import_numpy():
    np.matmul
    print("Success")

import_numpy()
import_numpy = types.FunctionType(import_numpy.__code__, {})
try:
    import_numpy()
except Exception as e:
    print(str(e))

Success
name 'np' is not defined


In [ ]:
# Unsloth's create_locked_down_function executes the generated code with a
# restricted set of builtins and a restricted importer, so a reward hacking
# completion cannot reach os, subprocess, sockets or the filesystem.
from unsloth import create_locked_down_function

# Countering Reward Hacking 3: Stop caching
We can stop the RL algorithm from using cached data by wiping the cache with a large fake matrix. We also have to benchmark carefully with multiple loops and turns.

We also add a **timer** to not make the algorithm go in an endless loop.

In [ ]:
import os, gc, time, statistics
import signal
from contextlib import contextmanager
class TimeoutError(Exception): pass

@contextmanager
def time_limit(seconds):
    def _handler(signum, frame):
        raise TimeoutError(f"Timed out after {seconds}s")
    old = signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, seconds)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, old)

class Benchmarker:
    def __init__(self, trials = 3, loops = 1, timeout = 30):
        self.buffer = np.zeros(2 * 1024 * 1024 * 1024, dtype = np.uint8)
        self.trials = trials
        self.loops = loops
        assert timeout > 0 # Cannot be 0 since it won't work!
        self.timeout = timeout
    def thrash(self):
        # Edit the buffer to wipe cache lines
        self.buffer ^= 1
        return int(self.buffer[::4096].sum())

    def benchmark(self, function, arguments):
        assert len(arguments) == self.loops
        samples = []
        exceptions = []
        timed_out = 0
        for _ in range(self.trials):
            gc.collect(); gc.disable(); self.thrash()
            t_start = time.perf_counter_ns()
            for i in range(self.loops):
                try:
                    with time_limit(self.timeout):
                        function(*arguments[i])
                except TimeoutError as e:
                    timed_out += 1
                except Exception as e:
                    exceptions.append(str(e))
            t_end = time.perf_counter_ns()
            gc.enable()
            samples.append((t_end - t_start) // max(1, self.loops))
        return {
            "median_ns": int(statistics.median(samples)),
            "mean_ns": int(statistics.fmean(samples)),
            "stdev_ns": int(statistics.pstdev(samples) if len(samples) > 1 else 0),
            "exceptions" : exceptions,
            "timeouts" : timed_out,
        }

For example we use our matmul kernel we had, and benchmark it with a 10 second delay:

In [ ]:
A, A_list, B, B_list = generate_random_matrices(seed = 0, n = 256)
Benchmarker(trials = 1, timeout = 10).benchmark(output_function["matmul"], [(A_list, B_list)])

{'median_ns': 32413305,
 'mean_ns': 32413305,
 'stdev_ns': 0,
 'exceptions': [],
 'timeouts': 0}

# Data & RL task setup

We now have to create a prompt to the model for which it will do some task. For our matrix multiply example, we use the below:

In [ ]:
prompt = """
Create a new fast matrix multiplication function using only native Python code.
You are given a list of list of numbers.
Output your new function in backticks using the format below:
```python
def matmul(A, B):
    return ...
```
""".strip()
print(prompt)

Create a new fast matrix multiplication function using only native Python code.
You are given a list of list of numbers.
Output your new function in backticks using the format below:
```python
def matmul(A, B):
    return ...
```


First, let's prompt GPT-OSS without RL and see how it goes:

In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("xpu"),
    temperature = 1.0,
    max_new_tokens = 512,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-11-10

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Create a new fast matrix multiplication function using only native Python code.
You are given a list of list of numbers.
Output your new function in backticks using the format below:
```python
def matmul(A, B):
    return ...
```<|end|><|start|>assistant<|channel|>analysis<|message|>We need to provide a quick naive implementation or maybe use Strassen? But only native Python. Provide efficient maybe using list comprehension and sum. Just implement standard O(n^3). Show code.<|end|><|start|>assistant<|channel|>final<|message|>Here is a concise, pure‑Python implementation that multiplies two matrices represented as lists of lists:

```python
def matmul(A, B):
    """
    Fast matrix multiplication using only native Pyt

# Reward functions

We now design the `extract_function` function which simply extracts the function wrapped in 3 backticks.

And 4 reward functions:

1. `function_works` which rewards the model if the strategy is a valid Python function.
2. `no_cheating` which checks if the function imported other modules, and if it did, we penalize it.
3. `correctness_check` which checks if the kernel was correct or wrong - it shouldn't generate gibberish!
4. `speed_check` checks the performance relative to Numpy matmul directly.

In [ ]:
def extract_function(text):
    if text.count("```") >= 2:
        first = text.find("```") + 3
        second = text.find("```", first)
        fx = text[first : second].strip()
        fx = fx.removeprefix("python\n")
        fx = fx[fx.find("def"):]
        if fx.startswith("def matmul(A, B):"): return fx
    return None
print(extract_function(prompt))

def matmul(A, B):
    return ...


Below is our `function_works` reward function which uses Python's `exec` but guarded by not allowing leakage of local and global variables. We can also use `check_only_stdlib_imports` first to check if there are errors before even executing the function:

In [ ]:
ok, info = check_only_stdlib_imports("def a")
ok, info

(False,
 {'error': "SyntaxError: expected '(' (<unknown>, line 1)",
  'stdlib': [],
  'non_stdlib': [],
  'relative_imports': 0})

In [ ]:
def function_works(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        print(function)
        if function is not None:
            ok, info = check_only_stdlib_imports(function)
        if function is None or "error" in info:
            score = -2.0
        else:
            try:
                new_matmul = create_locked_down_function(function)
                score = 1.0
            except:
                score = -0.5
        scores.append(score)
    return scores

`no_cheating` checks if the function cheated since it might have imported Numpy or Torch optimized code.

In [ ]:
def no_cheating(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        if function is not None:
            ok, info = check_only_stdlib_imports(function)
        else:
            ok = False
        scores.append(1.0 if ok else -20.0) # Penalize heavily!
    return scores

Next `correctness_check` checks if the kernel was correct. We want to penalize if the absolute error is larger than 1, and if the mean squared error is somewhat bigger then machine epsilon.

We have to execute the code now!

In [ ]:
np.finfo(np.float64).eps

np.float64(2.220446049250313e-16)

In [ ]:
def correctness_check(completions, **kwargs):
    scores = []
    # Generate some random matrices of size less than 128
    A, A_list, B, B_list = generate_random_matrices(seed = np.random.randint(10000), n = 128)
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        if function is not None:
            ok, info = check_only_stdlib_imports(function)
        if function is None or "error" in info:
            scores.append(0)
            continue
        try:
            new_matmul = create_locked_down_function(function)
        except:
            scores.append(0)
            continue
        try:
            pred = new_matmul(A_list.copy(), B_list.copy())
        except:
            # Failed!
            scores.append(-2.0)
            continue
        true = np.matmul(A, B)
        amax_error, mse_error = calculate_difference(pred, true)

        # Check correctness and score!
        machine_epsilon = 100*np.finfo(np.float64).eps
        if   amax_error >= 3:   score = -3.0
        elif amax_error >= 2:   score = -2.5
        elif amax_error >= 1:   score = -2.0
        elif amax_error >= 0.5: score = -1.0
        elif amax_error >= 100*machine_epsilon: score = 0.0
        elif amax_error >= machine_epsilon: score = 1.0
        else: score = 3.0

        if   mse_error >= 3:   score += -3.0
        elif mse_error >= 2:   score += -2.5
        elif mse_error >= 1:   score += -2.0
        elif mse_error >= 0.5: score += -1.0
        elif mse_error >= 100*machine_epsilon: score += 0.0
        elif mse_error >= machine_epsilon: score += 1.0
        else: score += 3.0
        scores.append(score)
    return scores

Finally our benchmarking function for `speed_check`! We shall limit the timer to 10 seconds and do 3 trials.

In [ ]:
A, A_list, B, B_list = generate_random_matrices(seed = 0, n = 256)
benchmarker = Benchmarker(trials = 3, timeout = 10)
numpy_results = benchmarker.benchmark(np.matmul, [(A, B)])
numpy_results

{'median_ns': 121473,
 'mean_ns': 133093,
 'stdev_ns': 19524,
 'exceptions': [],
 'timeouts': 0}

In [ ]:
new_matmul = create_locked_down_function(extract_function(prompt))
new_results = benchmarker.benchmark(new_matmul, [(A_list, B_list)])
new_results

{'median_ns': 49581,
 'mean_ns': 50597,
 'stdev_ns': 1678,
 'exceptions': [],
 'timeouts': 0}

We can take the difference and do a negative sign for slower ones. If the ratio is less than 1 (ie faster, we shall invert it!)

In [ ]:
negative = -(new_results["median_ns"] / numpy_results["median_ns"]) / 100
positive = +(numpy_results["median_ns"] / new_results["median_ns"]) / 100
reward = negative if new_results["median_ns"] >= numpy_results["median_ns"] else positive
reward

0.024499909239426396

In [ ]:
new_results["median_ns"] = 3
numpy_results["median_ns"] = 1000
negative = -(new_results["median_ns"] / numpy_results["median_ns"]) / 100
positive = +(numpy_results["median_ns"] / new_results["median_ns"]) / 100
reward = negative if new_results["median_ns"] >= numpy_results["median_ns"] else positive
reward

3.333333333333333

In [ ]:
import gc
def speed_check(completions, **kwargs):
    scores = []
    # Generate some random matrices of size less than 256
    A, A_list, B, B_list = generate_random_matrices(seed = np.random.randint(10000), n = 256)
    numpy_results = benchmarker.benchmark(np.matmul, [(A, B)])
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        if function is not None:
            ok, info = check_only_stdlib_imports(function)
        if function is None or "error" in info:
            scores.append(0)
            continue
        try:
            new_matmul = create_locked_down_function(function)
        except:
            scores.append(0)
            continue
        new_results = benchmarker.benchmark(new_matmul, [(A_list.copy(), B_list.copy())])

        # Get score and clip to -10, 10
        negative = -(new_results["median_ns"] / numpy_results["median_ns"]) / 100
        positive = +(numpy_results["median_ns"] / new_results["median_ns"]) / 100
        score = negative if new_results["median_ns"] >= numpy_results["median_ns"] else positive
        if score >= 10:  score = 10
        if score <= -10: score = -10
        scores.append(score)
    # Free memory to counteract OOMs
    gc.collect()
    torch.xpu.empty_cache()
    return scores

We create the dataset which includes a replica of our prompt. Remember to add reasoning effort of low!

In [ ]:
from datasets import Dataset
dataset = Dataset.from_list([{"prompt" : [{"role": "user", "content": prompt.strip()}], "answer" : 0, "reasoning_effort": "low"}]*1000)
maximum_length = len(tokenizer.apply_chat_template([{"role":"user", "content":prompt.strip()}], add_generation_prompt = True, tokenize = True))
print(maximum_length)
dataset[0]

116


{'prompt': [{'role': 'user',
   'content': 'Create a new fast matrix multiplication function using only native Python code.\nYou are given a list of list of numbers.\nOutput your new function in backticks using the format below:\n```python\ndef matmul(A, B):\n    return ...\n```'}],
 'answer': 0,
 'reasoning_effort': 'low'}

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations! We also support GSDP, GAPO, Dr GRPO and more! Go to our docs https://unsloth.ai/docs/ for more info!

In [ ]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

Unsloth: GRPO now defaults to TRL's loss_type = 'dapo' with beta = 0.0. Set loss_type = 'bnpo', beta = 0.001 for Unsloth's previous default.
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        function_works,
        no_cheating,
        correctness_check,
        speed_check,
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)

And let's train the model!

**NOTE** This training run might take quite long, so please wait!

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 4
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 46,227,456 of 20,960,984,640 (0.22% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072}. If this is not desired, please set these values explicitly.


def matmul(A, B):
    """Multiply two matrices A and B using pure Python."""
    # Ensure there is dimension compatibility
    m, n = len(A), len(A[0])
    p, q = len(B), len(B[0])
    if n != p:
        raise ValueError("Incompatible dimensions: A is {}x{}, B is {}x{}".format(m, n, p, q))

    # Initialize result matrix with zeros
    result = [[0] * q for _ in range(m)]

    # Compute the product
    for i in range(m):
        Ai = A[i]
        Ri = result[i]
        for k in range(n):
            aik = Ai[k]
            Bk = B[k]
            for j in range(q):
                Ri[j] += aik * Bk[j]
    return result
def matmul(A, B):
    """
    Multiply two matrices A and B using the standard triple‑loop algorithm.
    The function accepts lists of lists containing numbers (int or float)
    and returns the product matrix.

    Parameters
    ----------
    A : list[list[float]]
        Left matrix of shape (m, n)
    B : list[list[float]]
        Right matrix of shape (n, p)

    Re

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / function_works / mean,rewards / function_works / std,rewards / no_cheating / mean,rewards / no_cheating / std,rewards / correctness_check / mean,rewards / correctness_check / std,rewards / speed_check / mean,rewards / speed_check / std
1,0.000000,4.642185,0.012557,324.000000,213.000000,435.000000,0.000000,324.000000,213.000000,435.000000,0.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.000000,-1.357815,0.012557
2,0.000000,-8.521194,19.061911,411.000000,171.000000,651.000000,0.500000,171.000000,171.000000,171.000000,0.000000,-0.500000,2.121320,-9.500000,14.849242,2.000000,2.828427,-0.521194,0.737079
3,0.000000,3.261226,0.130274,237.500000,210.000000,265.000000,0.000000,237.500000,210.000000,265.000000,0.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.000000,-2.738774,0.130274
4,0.000000,4.466444,0.243897,330.000000,214.000000,446.000000,0.000000,330.000000,214.000000,446.000000,0.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.000000,-1.533556,0.243897
5,0.000000,2.382509,2.662269,402.500000,172.000000,633.000000,0.000000,402.500000,172.000000,633.000000,0.000000,0.250000,1.060660,1.000000,0.000000,2.000000,2.828427,-0.867491,1.226818
6,0.000000,3.675159,0.049431,242.500000,208.000000,277.000000,0.000000,242.500000,208.000000,277.000000,0.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.000000,-2.324841,0.049431
7,0.000000,2.797503,3.936330,240.500000,210.000000,271.000000,0.000000,240.500000,210.000000,271.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,4.242640,-0.202497,0.306311
8,0.000000,3.569474,0.452185,289.000000,282.000000,296.000000,0.000000,289.000000,282.000000,296.000000,0.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.000000,-2.430526,0.452185
9,0.000000,-1.258517,1.658918,222.000000,211.000000,233.000000,0.000000,222.000000,211.000000,233.000000,0.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.000000,-7.258516,1.658918
10,0.000000,5.832346,0.014364,355.500000,280.000000,431.000000,0.000000,355.500000,280.000000,431.000000,0.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.000000,-0.167654,0.014364


None
def matmul(A, B):
    # Get dimensions
    m = len(A)
    k = len(A[0]) if m > 0 else 0
    n = len(B[0]) if B else 0

    # Prepare result matrix with zeros
    C = [[0] * n for _ in range(m)]

    # Triple loop for multiplication
    for i in range(m):
        a_row = A[i]
        c_row = C[i]
        for j in range(n):
            b_col_sum = 0
            for p in range(k):
                b_col_sum += a_row[p] * B[p][j]
            c_row[j] = b_col_sum

    return C
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
def matmul(A, B):
    """Multiply two matrices A and B using pure Python.

    Parameters
    ----------
    A : list[list[float]]
        Left matrix of shape (m, n).
    B : list[list[float]]
        Right matrix of shape (n, p).

    Returns
    -------
    list[list[float]]
        Product matrix C of shape (m, p).
    """
    # Dimensions
    m = len(A)
    n = len(A[0]) if A else 0
    p = len(B[0]) if B else 0

    # Pre‑transpose

TrainOutput(global_step=100, training_loss=-1.2567782928796233e-07, metrics={'train_runtime': 40784.3865, 'train_samples_per_second': 0.005, 'train_steps_per_second': 0.002, 'total_flos': 0.0, 'train_loss': -1.2567782928796233e-07})

<a name="Inference"></a>
# Inference
Now let's try the model we just trained!

In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("xpu"),
    temperature = 1.0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-11-11

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Create a new fast matrix multiplication function using only native Python code.
You are given a list of list of numbers.
Output your new function in backticks using the format below:
```python
def matmul(A, B):
    return ...
```<|end|><|start|>assistant<|channel|>analysis<|message|>Need fast but native Python. Use list comprehension and zip. Provide function.<|end|><|start|>assistant<|channel|>final<|message|>```python
def matmul(A, B):
    # Number of rows in A and columns in B must match
    n = len(A)
    m = len(B[0])
    # Transpose B to make column access contiguous
    Bt = list(zip(*B))
    return [
        [sum(a * b for a, b in zip(row, col)) for col in Bt]
        for row in A
    ]
```
This uses only nat

<a name="Save"></a>
### Saving to float16 or MXFP4 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `mxfp4` for MXFP4 (OpenAI's GPT-OSS native precision). We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("gpt_oss_finetune_4bit", tokenizer, save_method = "mxfp4")
if False: model.push_to_hub_merged("repo_id/gpt_oss_finetune_4bit", tokenizer, token = "YOUR_HF_TOKEN", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>